[![Fixel Algorithms](https://i.imgur.com/AqKHVZ0.png)](https://fixelalgorithms.gitlab.io)

# AI Program

## Machine Learning - UnSupervised Learning - Dimensionality Reduction - MultiDimensional Scaling (MDS)

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 20/06/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0066DimensionalityReductionMDS.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning
from sklearn.manifold import ClassicalMDS, MDS

# Miscellaneous
from platform import python_version
import random

# Typing
from typing import Callable, Dict, List, Optional, Self, Set, Tuple, Union
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

EARTH_RADIUIS_KM = 6_371 #<! Mean radius of the Earth in kilometers (WGS84 standard)

In [ ]:
# Courses Packages

from DataVisualization import PlotScatterData, PlotScatterData3D

In [ ]:
# General Auxiliary Functions


## Dimensionality Reduction by MDS

The MDS is a family of transformation from $\mathbb{R}^{D} \to \mathbb{R}^{d}$ where commonly $d \ll D$.  
Given a set $\mathcal{X} = {\left\{ \boldsymbol{x}_{i} \in \mathbb{R}^{D} \right\}}_{i = 1}^{n}$ is builds the set $\mathcal{Z} = {\left\{ \boldsymbol{z}_{i} \in \mathbb{R}^{d} \right\}}_{i = 1}^{n}$ such that:

 - Non Metric MDS (Non Linear): Preserve the ranking of distances / dissimilarities of ${\left\{ \boldsymbol{x}_{i} \in \mathbb{R}^{D} \right\}}_{i = 1}^{n}$ in $\mathbb{R}^{d}$.
 - Metric MDS (Non Linear): Preserve the distances / dissimilarities of ${\left\{ \boldsymbol{x}_{i} \in \mathbb{R}^{D} \right\}}_{i = 1}^{n}$ in $\mathbb{R}^{d}$.
 - Classical MDS (Linear): Preserve the correlation of ${\left\{ \boldsymbol{x}_{i} \in \mathbb{R}^{D} \right\}}_{i = 1}^{n}$ in $\mathbb{R}^{d}$.

This notebook explores the use of MDS to recover points from a given distance matrix.


* <font color='brown'>(**#**)</font> Conceptually, MDS' input is the _Distance Matrix_ between all data samples.
* <font color='brown'>(**#**)</font> A closely related algorithm, yet much faster, is the: [FastMap: a fast algorithm for indexing, data-mining and visualization of traditional and multimedia datasets](https://doi.org/10.1184/R1/6605624). See [FastMap Dimension Reduction in the Browser](https://lucasb.eyer.be/articles/fastmap.html) for review of the algorithm.

In [ ]:
# Parameters

# Data
settelementDataUrl     = r'https://huggingface.co/datasets/Royi/DataSets/resolve/main/Israel1000Settlements.parquet'
settelementDistanceUrl = r'https://huggingface.co/datasets/Royi/DataSets/resolve/main/Israel1000SettlementsGeodesicDistances.parquet'

# Model
lowDim = 2

# Visualization
minMakrSize = 1
maxMakrSize = 20

## Generate / Load Data

This notebooks uses data about Israel's largest settlements.  

In [ ]:
# Generate Data

dfSettlements = pd.read_parquet(settelementDataUrl)
dfDistances   = pd.read_parquet(settelementDistanceUrl) #<! [Kilo Meters] Geodesic distances between the settlements

print(f'The meta data shape: {dfSettlements.shape}')
print(f'The distance data shape: {dfDistances.shape}')

In [ ]:
# The Meta Data
# Show the data in the Dataframe
dfSettlements

In [ ]:
# The Meta Data
# Show the data in the Dataframe
dfDistances

### Plot Data

In [ ]:
# Plot the Data

dfSettlementsPlot = dfSettlements.copy()

minPop = dfSettlements['population'].min()
maxPop = dfSettlements['population'].max()

dfSettlementsPlot['MarkerSize'] = (dfSettlementsPlot['population'] - minPop) / (maxPop - minPop) * (maxMakrSize - minMakrSize) + minMakrSize #<! For display purposes

hF = px.scatter_map(
    dfSettlementsPlot,
    lat = 'lat',
    lon = 'lon',
    hover_name = 'name',
    hover_data = {
        'population': ':,',
        'lat': ':.4f',
        'lon': ':.4f',
        'MarkerSize': False,
    },
    size = 'MarkerSize',
    size_max = 20,
    zoom = 6.4,
    center = {'lat': 31.8, 'lon': 35.0},
    height = 700,
)

hF.update_traces(marker = {'opacity': 0.75})
hF.update_layout(
    mapbox_style = 'open-street-map',
    margin = {'l': 0, 'r': 0, 't': 35, 'b': 0},
    title = 'Israel Settlements',
)

hF.show()

## MDS - Recovering Data By the Distance Matrix

The MDS recover points in the space of $\mathbb{R}^{d}$ based on a _dissimilarity_ / _distance_ matrix of teh high dimensional data.  
It means if one set $d - D$ the algorithm basically recovers the original data up to translation, rotation and reflection (Operators which preserve distance).

* <font color='brown'>(**#**)</font> For Euclidean distance there is a closed form solution (As with the PCA).

In [ ]:
# The Distance Table
# The distance table is a square matrix that contains the pairwise distances between settlements.
# Each entry (i, j) in the matrix represents the distance from settlement i to settlement j.
# The diagonal entries are zero since the distance from a settlement to itself is zero.
# The distance is in Kilo Meters and calculated by a Geodesic Distance (Matching Google Maps distance).

dfDistances

In [ ]:
# The Distance Matrix
# Extract the distance matrix from the Data Frame.

#===========================Fill This===========================#
# 1. Extract the distance matrix from the DataFrame `dfDistances` into a NumPy array `mD`.
mD = ???
#===============================================================#

In [ ]:
# From Distance to Points
# Classical MDS estimates coordinates whose pairwise distances match the given distance matrix.

#===========================Fill This===========================#
# 1. Define the MDS model using `ClassicalMDS`.
# 2. Fit the model to the distance matrix `mD` and transform it to obtain the low dimensional representation `mZ`.
# !! Define the number of components and the metric to match the use case.
oMdsDr = ???
mZ     = ???
#===============================================================#

In [ ]:
# Plot the Results
# This results are the estimated coordinates of the settlements in a 2D space.
# The should match the original coordinates of the settlements up to a rotation, translation, and reflection.

hF = go.Figure()
hF.add_trace(go.Scatter(
    x = mZ[:, 0],
    y = mZ[:, 1],
    mode = 'markers',
    marker = {'color': 'rgba(0, 0, 0, 0.45)', 'size': 7.5},
    text = dfSettlements['name'],
    customdata = dfSettlements[['population']],
    hovertemplate = '%{text}<br>Population: %{customdata[0]:,}<extra></extra>',
    showlegend = False,
))
hF.show()

### Convert Longitude / Latitude to Cartesian Coordinates

![](https://i.imgur.com/yMOBNMp.png)
<!-- ![](https://i.postimg.cc/xdRw4wQc/image.png) -->

The settlement locations are given by longitude / latitude angles on the surface of the Earth.  
To compare them with the MDS output, we convert them into local planar coordinates measured in kilometers around the mean location of the data.

Let $R$ be the Earth radius, $\phi$ the latitude and $\lambda$ the longitude in radians.  
For a small region around $(\bar{\phi}, \bar{\lambda})$, the local planar approximation is:

$$
x \approx R \cos(\bar{\phi}) \left(\lambda - \bar{\lambda}\right)
$$

$$
y \approx R \left(\phi - \bar{\phi}\right)
$$

The cosine factor appears only in the longitude coordinate.  
Latitude measures motion north / south along a meridian, whose radius is approximately the Earth radius $R$, hence $R \Delta \phi$.  
Longitude measures motion east / west along a circle parallel to the equator. At latitude $\phi$, that circle has radius $R \cos(\phi)$, so the same longitude difference becomes shorter as we move away from the equator.

In other words, one degree of latitude is almost the same distance everywhere, while one degree of longitude is scaled by $\cos(\phi)$

In [ ]:
# Planar / Cartesian Coordinates for the Settlements
# Build a local planar reference from (lat, lon) in units of kilometers.

#===========================Fill This===========================#
# 1. Extract the latitude in radians from `dfSettlements`.
# 2. Extract the longitude in radians from `dfSettlements`.
# 3. Compute the mean latitude and longitude in radians (Center point of the map).
# 4. Convert the latitude and longitude differences to kilometers around the mean.
# !! You may find `np.deg2rad` useful.
# !! You may find `EARTH_RADIUIS_KM` useful.
vLatRad = ??? #<! Latitude in radians
vLonRad = ??? #<! Longitude in radians
latMean = ??? #<! Mean latitude in radians (Center point of the map)
lonMean = ??? #<! Mean longitude in radians (Center point of the map)

mXMap = np.column_stack((
    ???, #<! Convert longitude differences to kilometers with adjustment for latitude
    ???, #<! Convert latitude differences to kilometers
))
#===============================================================#

### Recover Rotation and Translation

Assume two matched sets of points in $\mathbb{R}^{2}$:

 - $\mathcal{X} = {\left\{ \boldsymbol{x}_{i} \right\}}_{i = 1}^{n}$.
 - $\mathcal{Z} = {\left\{ \boldsymbol{z}_{i} \right\}}_{i = 1}^{n}$.

Both sets describe the same geometry up to translation and an orthogonal transformation, namely rotation and possibly reflection:

$$
\boldsymbol{x}_{i} \approx \boldsymbol{R}^{T} \boldsymbol{z}_{i} + \boldsymbol{t}
$$

Using _row wise_ data matrices $\boldsymbol{X}, \boldsymbol{Z} \in \mathbb{R}^{n \times 2}$:

$$
\boldsymbol{X} \approx \boldsymbol{Z} \boldsymbol{R} + t \boldsymbol{1} \boldsymbol{1}^{T}, \qquad
\boldsymbol{R}^{T} \boldsymbol{R} = \boldsymbol{I}
$$

The translation is removed by centering both point sets around their means:

$$
\bar{\boldsymbol{x}} = \frac{1}{n} \sum_{i = 1}^{n} \boldsymbol{x}_{i}, \qquad
\bar{\boldsymbol{z}} = \frac{1}{n} \sum_{i = 1}^{n} \boldsymbol{z}_{i}
$$

$$
\boldsymbol{X}_{0} = \boldsymbol{X} - \boldsymbol{1} \bar{\boldsymbol{x}}^{T}, \qquad
\boldsymbol{Z}_{0} = \boldsymbol{Z} - \boldsymbol{1} \bar{\boldsymbol{z}}^{T}
$$

The optimal orthogonal matrix is recovered by the Orthogonal Procrustes problem:

$$
\hat{\boldsymbol{R}} = \arg \min_{\boldsymbol{R}} \left\| \boldsymbol{Z}_{0} \boldsymbol{R} - \boldsymbol{X}_{0} \right\|_{F}^{2}
\quad \text{s.t.} \quad
\boldsymbol{R}^{T} \boldsymbol{R} = \boldsymbol{I}
$$

If $\boldsymbol{Z}_{0}^{T} \boldsymbol{X}_{0} = \boldsymbol{U} \boldsymbol{\Sigma} \boldsymbol{V}^{T}$ is the SVD, then:

$$
\hat{\boldsymbol{R}} = \boldsymbol{U} \boldsymbol{V}^{T}
$$

Finally, the translation is recovered from the means:

$$
\hat{\boldsymbol{t}} = \bar{\boldsymbol{x}} - \hat{\boldsymbol{R}}^{T} \bar{\boldsymbol{z}}
$$

For the row-wise implementation below this is equivalent to:

$$
\hat{\boldsymbol{X}} = \left( \boldsymbol{Z} - \boldsymbol{1} \bar{\boldsymbol{z}}^{T} \right) \hat{\boldsymbol{R}} + \boldsymbol{1} \bar{\boldsymbol{x}}^{T}
$$

Thus, once the point correspondence is known, the rotation / reflection and translation that best align the two point sets are obtained in closed form.

In [ ]:
# Recover the Rotation and Translation
# Align the recovered MDS coordinates to the map coordinates by rotation / reflection only.

#===========================Fill This===========================#
# 1. Calculate the mean coordinate of the map.
# 2. Calculate the mean coordinate of the MDS recovery.
# 3. Center the map coordinates and the MDS recovery by subtracting their respective means.
# 4. Use the orthogonal Procrustes analysis to find the optimal rotation matrix `mR`.
# 5. Recover the aligned MDS coordinates by applying the rotation and adding back the mean of the map coordinates.
# !! You may find the function `sp.linalg.orthogonal_procrustes` useful for this step.

vXMean = ??? #<! Mean coordinate of the map
vZMean = ??? #<! Mean coordinate of the MDS recovery
mX0    = ??? #<! Center the map coordinates by subtracting the mean of the map coordinates
mZ0    = ??? #<! Center the MDS recovery by subtracting the mean of the MDS recovery
mR, _  = ??? #<! Solved \arg \min_R || mZ0 R - mX0 ||_F^2 subject to R^T R = I

mZAligned = ??? #<! Aligned MDS coordinates after applying the optimal rotation and adding back the mean of the map coordinates
#===============================================================#

mErr = mZAligned - mXMap
rmseKm = np.sqrt(np.mean(np.sum(np.square(mErr), axis = 1)))
print(f'Alignment RMSE: {rmseKm:.2f} [km]')

# Plot the original map coordinates and the MDS recovery after alignment.
vLineX = np.column_stack((mXMap[:, 0], mZAligned[:, 0], np.full(mXMap.shape[0], np.nan))).ravel()
vLineY = np.column_stack((mXMap[:, 1], mZAligned[:, 1], np.full(mXMap.shape[0], np.nan))).ravel()

In [ ]:
# Plot the Aligned Data
hF = go.Figure()
hF.add_trace(go.Scatter(
    x = vLineX,
    y = vLineY,
    mode = 'lines',
    line = {'color': 'rgba(0, 0, 0, 0.15)', 'width': 1},
    hoverinfo = 'skip',
    showlegend = False,
))
hF.add_trace(go.Scatter(
    x = mXMap[:, 0],
    y = mXMap[:, 1],
    mode = 'markers',
    name = 'Location from latitude / longitude',
    marker = {'size': 7, 'color': '#1f77b4', 'opacity': 0.8},
    text = dfSettlements['name'],
    hovertemplate = '%{text}<br>x: %{x:.2f} [km]<br>y: %{y:.2f} [km]<extra></extra>',
))
hF.add_trace(go.Scatter(
    x = mZAligned[:, 0],
    y = mZAligned[:, 1],
    mode = 'markers',
    name = 'Recovered from distances',
    marker = {'size': 7, 'color': '#ff7f0e', 'opacity': 0.75, 'symbol': 'x'},
    text = dfSettlements['name'],
    hovertemplate = '%{text}<br>x: %{x:.2f} [km]<br>y: %{y:.2f} [km]<extra></extra>',
))
hF.update_layout(
    title = 'Classical MDS Recovery from the Distance Matrix: Equal up to Rotation / Reflection',
    xaxis_title = 'x [km]',
    yaxis_title = 'y [km]',
    width = 850,
    height = 850,
)
hF.update_yaxes(scaleanchor = 'x', scaleratio = 1)
hF.show()

### Metric MDS

The Metric MDS tries to find a low level representation which preserved the pairwise distances of the high dimensional data.

* <font color='brown'>(**#**)</font> The optimization problem the _Metric MDS_ solves is highly _Non Convex_ and _Non Linear_.

In [ ]:
# From Distance to Points

# MDS estimates coordinates whose pairwise distances match the given distance matrix.
oMdsDr = MDS(
    n_components = lowDim,
    init = 'random', #<! Specifically not to use the `classical_mds` initialization which is the closed form solution of the Euclidean case
    n_init = 20,
    metric = 'precomputed',
    normalized_stress = 'auto',
    random_state = seedNum,
)
mZ = oMdsDr.fit_transform(mD)

In [ ]:
# Plot the Results
# This results are the estimated coordinates of the settlements in a 2D space.
# The should match the original coordinates of the settlements up to a rotation, translation, and reflection.

hF = go.Figure()
hF.add_trace(go.Scatter(
    x = mZ[:, 0],
    y = mZ[:, 1],
    mode = 'markers',
    marker = {'color': 'rgba(0, 0, 0, 0.45)', 'size': 7.5},
    text = dfSettlements['name'],
    customdata = dfSettlements[['population']],
    hovertemplate = '%{text}<br>Population: %{customdata[0]:,}<extra></extra>',
    showlegend = False,
))
hF.show()

In [ ]:
# Recover the Rotation and Translation

# Align the recovered MDS coordinates to the map coordinates by rotation / reflection only.
vZMean = np.mean(mZ, axis = 0)
mZ0    = mZ - vZMean
mR, _  = sp.linalg.orthogonal_procrustes(mZ0, mX0)

mZAligned = (mZ0 @ mR) + vXMean

mErr = mZAligned - mXMap
rmseKm = np.sqrt(np.mean(np.sum(np.square(mErr), axis = 1)))
print(f'Alignment RMSE: {rmseKm:.2f} [km]')

# Plot the original map coordinates and the MDS recovery after alignment.
vLineX = np.column_stack((mXMap[:, 0], mZAligned[:, 0], np.full(mXMap.shape[0], np.nan))).ravel()
vLineY = np.column_stack((mXMap[:, 1], mZAligned[:, 1], np.full(mXMap.shape[0], np.nan))).ravel()

In [ ]:
# Plot Results
hF = go.Figure()
hF.add_trace(go.Scatter(
    x = vLineX,
    y = vLineY,
    mode = 'lines',
    line = {'color': 'rgba(0, 0, 0, 0.15)', 'width': 1},
    hoverinfo = 'skip',
    showlegend = False,
))
hF.add_trace(go.Scatter(
    x = mXMap[:, 0],
    y = mXMap[:, 1],
    mode = 'markers',
    name = 'Location from latitude / longitude',
    marker = {'size': 7, 'color': '#1f77b4', 'opacity': 0.8},
    text = dfSettlements['name'],
    hovertemplate = '%{text}<br>x: %{x:.2f} [km]<br>y: %{y:.2f} [km]<extra></extra>',
))
hF.add_trace(go.Scatter(
    x = mZAligned[:, 0],
    y = mZAligned[:, 1],
    mode = 'markers',
    name = 'Recovered from distances',
    marker = {'size': 7, 'color': '#ff7f0e', 'opacity': 0.75, 'symbol': 'x'},
    text = dfSettlements['name'],
    hovertemplate = '%{text}<br>x: %{x:.2f} [km]<br>y: %{y:.2f} [km]<extra></extra>',
))
hF.update_layout(
    title = 'Metric MDS Recovery from the Distance Matrix: Equal up to Rotation / Reflection',
    xaxis_title = 'x [km]',
    yaxis_title = 'y [km]',
    width = 850,
    height = 850,
)
hF.update_yaxes(scaleanchor = 'x', scaleratio = 1)
hF.show()

* <font color='red'>(**?**)</font> How come the _Metric MDS_ results are worse than the _Classical MDS_ results?